# LC 79 — Word Search
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Backtracking
**Pattern:** DFS + Backtrack on Grid — Mark and Unmark

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> DFS from every cell
that matches the first character. Mark visited
cells temporarily to prevent reuse in the same
path. Unmark (backtrack) when the path fails or
completes.
</div>

## Official Problem Statement

Given an `m x n` grid of characters `board` and
a string `word`, return `true` if `word` exists
in the grid.

The word can be constructed from letters of
sequentially adjacent cells, where adjacent cells
are horizontally or vertically neighbouring. The
same letter cell may **not** be used more than
once.

**Example 1:**
```
Input:
  board = [["A","B","C","E"],
            ["S","F","C","S"],
            ["A","D","E","E"]]
  word = "ABCCED"
Output: true
```
**Example 2:**
```
Input: same board, word = "SEE"
Output: true
```
**Example 3:**
```
Input: same board, word = "ABCB"
Output: false
```

**Constraints:**
- `m == board.length`, `n == board[i].length`
- `1 <= m, n <= 6`
- `1 <= word.length <= 15`
- `board` and `word` consist of only uppercase
  English letters

## What This Is Actually Asking

Find a path through the grid that spells out the
word letter by letter, moving only up/down/left/
right. You cannot revisit a cell in the same path.
Return True if any such path exists.

## Walk Through an Example by Hand

```
board = [["A","B","C","E"],
         ["S","F","C","S"],
         ["A","D","E","E"]]
word = "ABCCED"

Start: scan for 'A' -> (0,0) and (2,0)

DFS from (0,0): letter A matches word[0]
  Mark (0,0) = '#'
  Try neighbours for word[1]='B':
    (0,1)='B' matches!
    Mark (0,1)='#'
    Try for word[2]='C':
      (0,2)='C' matches!
      Mark (0,2)='#'
      Try for word[3]='C':
        (1,2)='C' matches!
        Mark (1,2)='#'
        Try for word[4]='E':
          (2,2)='E' matches!
          Mark (2,2)='#'
          Try for word[5]='D':
            (2,1)='D' matches!  index==len(word) -> True!
```

## The Picture

```
board = A  B  C  E
        S  F  C  S
        A  D  E  E

Path for "ABCCED":
  A(0,0) -> B(0,1) -> C(0,2) -> C(1,2) -> E(2,2) -> D(2,1)
  [*] [*] [*]  E
   S   F  [*]  S
   A  [*] [*]  E

Backtracking rule:
  dfs(r, c, idx):           # idx = position in word
    if idx == len(word): return True
    if out-of-bounds or board[r][c] != word[idx]: return False

    temp = board[r][c]
    board[r][c] = '#'       # MARK as visited

    found = any of 4 neighbours with idx+1

    board[r][c] = temp      # UNMARK (backtrack)
    return found

Marking with '#' prevents reuse in same path.
Unmarking restores the cell for other paths.
```

## When To Use This Pattern

- When searching for a path in a grid, think
  **DFS + backtrack — mark cell, recurse, unmark**
- When cells cannot be reused in a path, think
  **temporarily overwrite with '#', restore after**
- When idx reaches word length, think
  **found — return True immediately**
- When out of bounds or mismatch, think
  **return False — prune this branch**

## The Approach

Scan every cell for the first character. From each
match, run DFS passing the current index into the
word. At each DFS call: if the index equals the
word length the word is found. If out of bounds or
the character does not match, return False. Mark
the cell, recurse in 4 directions with index+1,
then restore the cell. Return True if any direction
succeeds.

In [ ]:
from typing import List  # type hints for the solution

In [ ]:
def test_harness(func):
    board1 = [["A","B","C","E"],
              ["S","F","C","S"],
              ["A","D","E","E"]]

    tests = [
        # (board, word, expected)
        (board1, "ABCCED", True),
        (board1, "SEE",    True),
        (board1, "ABCB",   False),  # can't reuse B
        ([["a"]], "a",     True),   # single cell match
        ([["a"]], "b",     False),  # single cell miss
        ([["A","B"],["C","D"]], "ABDC", True),
        ([["A","B"],["C","D"]], "ABCD", False), # D not adj C
    ]

    passed = 0
    for i, (board, word, expected) in enumerate(tests):
        b = [r[:] for r in board]
        result = func(b, word)
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"word={word!r} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def exist(
    board: List[List[str]], word: str
) -> bool:
    """
    Return True if word can be traced in the grid.

    DFS+backtrack: dfs(r, c, idx). Base: idx==len
    -> True. Guard: OOB or mismatch -> False. Mark
    board[r][c]='#'; recurse 4 dirs with idx+1;
    restore board[r][c]. Return True if any dir succeeds.

    Time:  O(m*n * 4^L) where L=word length
    Space: O(L) — recursion depth
    """
    pass


# Quick debug — run this cell while building
b = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]]
print(exist([r[:] for r in b], "ABCCED"))  # True
print(exist([r[:] for r in b], "SEE"))     # True
print(exist([r[:] for r in b], "ABCB"))    # False
print(exist([["a"]], "a"))                  # True

In [ ]:
# Uncomment and run when solution is ready
# test_harness(exist)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force all paths | O(m*n * 4^L) | O(L) |
| DFS + backtrack | O(m*n * 4^L) | O(L) |

Backtracking prunes failed paths early. In the
worst case all cells match every character but
the last — the complexity is still bounded by
the decision tree height L.

## Real World Connection

At Citi, the log pattern matcher searches a 2D
event grid (server × time) for a specific sequence
of correlated events — e.g., CPU spike followed
by network timeout followed by restart, each in
adjacent time slots.
Word Search on the event grid finds this pattern
path in O(m*n * 4^L) with backtracking that prunes
dead ends early.
On AWS CloudWatch, the log insights query engine
uses a similar DFS pattern to trace correlated
events across a structured log matrix.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra